In [1]:
import boto3
from collections import defaultdict

# ========= 在这里填入你的 key =========
ACCESS_KEY = "f04c088e-d96f-4944-a048-8e419c0bd524"
SECRET_KEY = "0_STHVfnT0CLigISYj9Oo0SIWVVpC9vO"
# ====================================

ENDPOINT = "https://files.massive.com"
BUCKET = "flatfiles"
PREFIX = ""   # 测试建议先 "stocks/"

# 中文映射（已覆盖你截图里的所有 asset）
ASSET_CN = {
    "us_stocks_sip": "美股全市场合并行情(SIP)",
    "us_options_opra": "美股期权(OPRA合并行情)",
    "us_futures_cme": "CME期货",
    "us_futures_cbot": "CBOT期货",
    "us_futures_nymex": "NYMEX期货",
    "us_futures_comex": "COMEX期货",
    "us_indices": "美国指数",
    "global_forex": "全球外汇",
    "global_crypto": "全球加密货币",
}

def human_size(n):
    for unit in ["B","KB","MB","GB","TB","PB"]:
        if n < 1024:
            return f"{n:.2f}{unit}"
        n /= 1024
    return f"{n:.2f}EB"


s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY
)

paginator = s3.get_paginator("list_objects_v2")

asset_bytes = defaultdict(int)
asset_files = defaultdict(int)

total_bytes = 0
total_files = 0
page_count = 0

print("Listing bucket... (会比较慢，耐心等)")

for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    page_count += 1
    
    for obj in page.get("Contents", []):
        key = obj["Key"]
        size = obj["Size"]

        asset = key.split("/")[0]
        asset_bytes[asset] += size
        asset_files[asset] += 1

        total_bytes += size
        total_files += 1

    if page_count % 50 == 0:
        print(f"processed {page_count} pages | files={total_files:,}")

print("\n===== RESULT =====")

print(f"{'asset':15s} {'中文含义':20s} {'大小(GB)':>12s} {'文件数':>12s} {'占比':>8s}")

for asset in sorted(asset_bytes, key=asset_bytes.get, reverse=True):
    gb = asset_bytes[asset] / 1024**3
    pct = asset_bytes[asset] / total_bytes * 100
    cn = ASSET_CN.get(asset, "未知类型")
    print(f"{asset:15s} {cn:20s} {gb:12.2f} {asset_files[asset]:12,d} {pct:7.2f}%")

print("\nTOTAL:", human_size(total_bytes), total_files, "files")

Listing bucket... (会比较慢，耐心等)
processed 50 pages | files=49,716
processed 100 pages | files=99,592

===== RESULT =====
asset           中文含义                       大小(GB)          文件数       占比
us_options_opra 美股期权(OPRA合并行情)           94055.18        9,864   76.73%
us_stocks_sip   美股全市场合并行情(SIP)           15557.60       22,608   12.69%
us_futures_cme  CME期货                     5659.76       18,011    4.62%
us_futures_nymex NYMEX期货                   2665.66       18,011    2.17%
us_indices      美国指数                      1910.30        2,379    1.56%
us_futures_cbot CBOT期货                    1603.60       18,011    1.31%
global_forex    全球外汇                       615.60       15,891    0.50%
us_futures_comex COMEX期货                    405.43       18,011    0.33%
global_crypto   全球加密货币                     105.50       13,512    0.09%

TOTAL: 119.71TB 136298 files


In [1]:
import pandas as pd
pd.read_parquet('/home/yluel/share/projects/massive_parquet/us_stocks_sip/quotes_v1/2026/02/2026-02-04.parquet')

,ticker,ask_exchange,ask_price,ask_size,bid_exchange,bid_price,bid_size,conditions,indicators,participant_timestamp,sequence_number,sip_timestamp,tape,trf_timestamp
0,A,8.0,0.00,0.0,8.0,0.0,0.0,"1,81",NaN,1.770196e+18,389.0,1.770196e+18,1.0,0.0
1,A,19.0,0.00,0.0,19.0,0.0,0.0,"1,81",NaN,1.770196e+18,489.0,1.770196e+18,1.0,0.0
2,A,20.0,0.00,0.0,20.0,0.0,0.0,"1,81",NaN,1.770196e+18,508.0,1.770196e+18,1.0,0.0
3,A,8.0,168.00,100.0,8.0,0.0,0.0,"1,81",NaN,1.770196e+18,699.0,1.770196e+18,1.0,0.0
4,A,8.0,154.17,2000.0,8.0,0.0,0.0,"1,81",NaN,1.770196e+18,702.0,1.770196e+18,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
791170636,ZZK,10.0,0.00,0.0,10.0,25.0,100.0,"1,81",NaN,1.770206e+18,412582.0,1.770206e+18,1.0,0.0
791170637,ZZK,10.0,0.00,0.0,10.0,0.0,0.0,"1,81",NaN,1.770207e+18,414805.0,1.770207e+18,1.0,0.0
791170638,ZZK,20.0,0.00,0.0,20.0,0.0,0.0,"1,81",NaN,1.770216e+18,8649170.0,1.770216e+18,1.0,0.0
791170639,ZZK,20.0,0.00,0.0,20.0,0.0,0.0,"1,81",NaN,1.770239e+18,120459535.0,1.770239e+18,1.0,0.0


In [ ]:
import boto3
from botocore.config import Config

# Initialize a session using your credentials
session = boto3.Session(
  aws_access_key_id='f04c088e-d96f-4944-a048-8e419c0bd524',
  aws_secret_access_key='0_STHVfnT0CLigISYj9Oo0SIWVVpC9vO',
)

# Create a client with your session and specify the endpoint
s3 = session.client(
  's3',
  endpoint_url='https://files.massive.com',
  config=Config(signature_version='s3v4'),
)

# Specify the bucket name
bucket_name = 'flatfiles'

# Specify the S3 object key name
object_key = 'flatfiles/us_stocks_sip/quotes_v1/2026/02/2026-02-04.csv.gz'

# Remove the bucket name (e.g. 'flatfiles/') prefix if present in object_key
if object_key.startswith(bucket_name + '/'):
  object_key = object_key[len(bucket_name + '/'):]

# Specify the local file name and path to save the downloaded file
local_file_name = object_key.split('/')[-1]  # e.g., '2025-06-12.csv.gz'
local_file_path = './' + local_file_name

# Print the file being downloaded
print(f"Downloading file '{object_key}' from bucket '{bucket_name}'...")

# Download the file
s3.download_file(bucket_name, object_key, local_file_path)

In [ ]:
import boto3
from botocore.config import Config

# Initialize a session using your credentials
session = boto3.Session(
  aws_access_key_id='f04c088e-d96f-4944-a048-8e419c0bd524',
  aws_secret_access_key='0_STHVfnT0CLigISYj9Oo0SIWVVpC9vO',
)

# Create a client with your session and specify the endpoint
s3 = session.client(
  's3',
  endpoint_url='https://files.massive.com',
  config=Config(signature_version='s3v4'),
)

# Specify the bucket name
bucket_name = 'flatfiles'

# Specify the S3 object key name
object_key = 'flatfiles/us_stocks_sip/quotes_v1/2026/02/2026-02-05.csv.gz'

# Remove the bucket name (e.g. 'flatfiles/') prefix if present in object_key
if object_key.startswith(bucket_name + '/'):
  object_key = object_key[len(bucket_name + '/'):]

# Specify the local file name and path to save the downloaded file
local_file_name = object_key.split('/')[-1]  # e.g., '2025-06-12.csv.gz'
local_file_path = './' + local_file_name

# Print the file being downloaded
print(f"Downloading file '{object_key}' from bucket '{bucket_name}'...")

# Download the file
s3.download_file(bucket_name, object_key, local_file_path)